# Intensive Care Unit Decision-Support Tool

**Group Project for Responsible Data Science**

**Submission Deadline: March 24, 2025**

This notebook provides a comprehensive analysis of ICU patient data to predict mortality, including advanced visualization, model explanations, and fairness evaluations.

## 1. Setup and Configuration

In [1]:
# Import standard libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

# Import custom modules
import data_processing as dp
import visualization as viz
import modeling as mdl
import utils
import config

# Set visualization style
viz.set_visualization_style()
plt.style.use('seaborn-v0_8-whitegrid')

# Create directories if they don't exist
os.makedirs(config.RESULTS_DIR, exist_ok=True)
os.makedirs(config.MODELS_DIR, exist_ok=True)

ImportError: cannot import name '_print_elapsed_time' from 'sklearn.utils' (C:\Users\klaem\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\__init__.py)

## 2. Data Loading and Preprocessing

In [ ]:
# Load data and data dictionary
df, description_dict = dp.load_data(config.DATA_DIR)
utils.print_dataset_info(df)

# Analyze missing values
missing_percentages, high_missing_cols = dp.analyze_missing_values(df)

# Plot missing values
viz.plot_missing_values(missing_percentages)

# Split data for modeling
X, y = dp.split_features_target(df, config.TARGET_COLUMN)
X_train, X_test, y_train, y_test = dp.create_train_test_split(
    X, y, test_size=config.TEST_SIZE, random_state=config.RANDOM_STATE
)

# Filter high missing columns and identify column types
X_train_filtered, X_test_filtered, high_missing_cols = dp.filter_high_missing_columns(
    X_train, X_test, threshold=config.MISSING_THRESHOLD_HIGH
)
numerical_cols, categorical_cols = dp.identify_column_types(X_train_filtered)

# Save processed data for later use
dp.save_processed_data(
    X_train_filtered, X_test_filtered, y_train, y_test,
    numerical_cols, categorical_cols, description_dict,
    os.path.join(config.RESULTS_DIR, 'processed_data.pkl')
)

## 3. Enhanced Data Exploration

In [ ]:
# Visualize target variable distribution
viz.plot_target_distribution(df)

In [ ]:
# Explore age distribution by mortality outcome
viz.plot_age_distribution_by_mortality(X_train_filtered, y_train)

In [ ]:
# Analyze key physiological parameters by mortality
physiological_vars = ['heart_rate', 'map_apache', 'resp_rate', 'temperature', 'wbc', 'creatinine']
viz.plot_physiological_vars_by_mortality(X_train_filtered, y_train, physiological_vars)

In [ ]:
# Visualize categorical features
key_categorical = ['ethnicity', 'apache_2_bodysystem', 'apache_3j_bodysystem', 'icu_type']
viz.plot_categorical_features_by_mortality(X_train_filtered, y_train, key_categorical)

In [ ]:
# Correlation analysis
viz.plot_correlation_matrix_of_clinical_features(X_train_filtered, y_train)

## 4. Model Training and Evaluation

In [ ]:
# Train models with different imputation strategies
pipelines, scores = mdl.train_and_evaluate_all_imputation_strategies(
    X_train_filtered, X_test_filtered, y_train, y_test,
    numerical_cols, categorical_cols, config.IMPUTATION_STRATEGIES, config.RF_PARAMS
)

In [ ]:
# Create enhanced confusion matrices for all models
for strategy, pipeline in pipelines.items():
    print(len(X_test_filtered))

    y_pred = pipeline.predict(X_test_filtered)
    viz.plot_enhanced_confusion_matrix(y_test, y_pred, strategy)

In [ ]:
# Compare ROC curves for different imputation strategies
viz.plot_roc_curves_comparison(
    y_test, {strategy: pipeline.predict_proba(X_test_filtered)[:, 1] for strategy, pipeline in pipelines.items()}
)

In [ ]:
# Compare Precision-Recall curves for different imputation strategies
viz.plot_precision_recall_curves_comparison(
    y_test, {strategy: pipeline.predict_proba(X_test_filtered)[:, 1] for strategy, pipeline in pipelines.items()}
)

In [ ]:
# Find the best imputation strategy
comparison_df, best_strategy = utils.compare_models({s: scores[s] for s in config.IMPUTATION_STRATEGIES}, 'roc_auc')

print(f"Best imputation strategy: {best_strategy}")
print(comparison_df)

# Save the best model for later use
best_model = pipelines[best_strategy]
with open(os.path.join(config.MODELS_DIR, f'best_model_{best_strategy}.pkl'), 'wb') as f:
    pickle.dump(best_model, f)

## 5. Feature Importance Analysis

In [ ]:
# Extract feature names and importances
feature_names = mdl.get_feature_names(pipelines[best_strategy].named_steps['preprocessor'])
feature_importances = pipelines[best_strategy].named_steps['classifier'].feature_importances_

# Plot top 20 features
importance_df = viz.plot_feature_importance(
    feature_names, feature_importances, top_n=20,
    title=f'Top 20 Feature Importances - {best_strategy.capitalize()} Imputation'
)

# Display feature importance table
print("Top 20 most important features:")
display(importance_df.head(20))

In [ ]:
# Try to add SHAP analysis if it's available
mdl.analyze_with_shap(best_model, X_test_filtered, feature_names)

## 6. Model Comparison with APACHE IV

In [ ]:
# Compare with APACHE IV prediction if available
apache_result = mdl.compare_with_apache_iv(X_test_filtered, y_test, best_model.predict_proba(X_test_filtered)[:, 1])

if apache_result is not None:
    apache_auc, our_auc = apache_result
    viz.plot_model_vs_apache_comparison(X_test_filtered, y_test, best_model, best_strategy, apache_auc, our_auc)

## 7. Fairness Analysis

In [ ]:
# Analyze model performance across different ethnic groups
best_model = pipelines[best_strategy]
y_pred = best_model.predict(X_test_filtered)
y_proba = best_model.predict_proba(X_test_filtered)[:, 1]

fairness_df, metrics_by_ethnicity = mdl.analyze_fairness_by_ethnicity(
    X_test_filtered, y_test, y_pred, y_proba
)

if fairness_df is not None:
    print("Fairness Metrics by Ethnicity:")
    display(fairness_df)
    
    # Plot fairness metrics
    viz.plot_fairness_metrics(fairness_df, metrics_by_ethnicity)

## 8. Analysis by Body System

In [ ]:
# Get a high-level overview of body system performance
body_system_df, body_system_metrics = mdl.analyze_performance_by_body_system(
    X_test_filtered, y_test, best_model
)

if body_system_df is not None:
    print("Model Performance by Body System:")
    display(body_system_df)
    
    # Plot performance by body system
    viz.plot_performance_by_body_system(body_system_df, body_system_metrics)

## 9. Additional Analysis: In-Depth Body System Modeling

In [ ]:
# Perform a more detailed analysis by body system
body_system_results = mdl.analyze_by_body_system(
    X_train_filtered, y_train, 
    X_test_filtered, y_test, 
    numerical_cols, categorical_cols, 
    best_strategy
)

if body_system_results is not None and len(body_system_results) > 0:
    viz.plot_body_system_analysis_results(body_system_results, body_system_df, feature_names)

## 10. Finding Optimal Classification Threshold

In [ ]:
# Find the optimal classification threshold for our best model
thresholds_analysis = mdl.analyze_classification_thresholds(best_model, X_test_filtered, y_test)
viz.plot_threshold_analysis_results(thresholds_analysis)